In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
train_dir = '/content/drive/MyDrive/autism-image-dataset/train'
valid_dir = '/content/drive/MyDrive/autism-image-dataset/valid'
test_dir  = '/content/drive/MyDrive/autism-image-dataset/test'

# Check if folders exist
for path in [train_dir, valid_dir, test_dir]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Folder not found: {path}")

In [ ]:
import torch
batch_size = 16
num_epochs = 10
learning_rate = 1e-4
image_size = 224
num_classes = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

valid_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
valid_dataset = datasets.ImageFolder(valid_dir, transform=valid_transform)
test_dataset  = datasets.ImageFolder(test_dir, transform=valid_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
import torch.nn as nn
from timm import create_model

class ViTASD(nn.Module):
    def __init__(self, backbone='vit_base_patch16_224', num_classes=2, drop_rate=0.0, drop_path_rate=0.1, input_size=224):
        super(ViTASD, self).__init__()
        self.backbone = create_model(
            backbone,
            pretrained=True,
            num_classes=num_classes,
            drop_rate=drop_rate,
            drop_path_rate=drop_path_rate,
            img_size=input_size
        )

    def forward(self, x):
        return self.backbone(x)

model = ViTASD(backbone='vit_base_patch16_224', num_classes=num_classes, input_size=image_size).to(device)
print(model)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
from tqdm import tqdm

best_val_acc = 0.0

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss /= total
    train_acc = correct / total

    # Validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= total
    val_acc = correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f} Train Acc: {train_acc:.4f} "
          f"Val Loss: {val_loss:.4f} Val Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/content/best_vit_autism_model.pth")


In [ ]:
model.load_state_dict(torch.load("/content/best_vit_autism_model.pth"))
model.eval()

correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

test_acc = correct / total
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
with torch.no_grad():
    for i, (images, labels) in enumerate(test_loader):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        for j in range(len(labels)):
            print(f"Image {i*batch_size + j}: True={labels[j].item()}, Predicted={preds[j].item()}")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=['non_autistic', 'autistic']))


In [ ]:
!pip install seaborn


In [ ]:
import torch
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
class_names = ['non_autistic', 'autistic']

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
import numpy as np

counts_true = [all_labels.count(0), all_labels.count(1)]
counts_pred = [all_preds.count(0), all_preds.count(1)]

x = np.arange(len(class_names))
width = 0.35

plt.figure(figsize=(6,5))
plt.bar(x - width/2, counts_true, width, label='True')
plt.bar(x + width/2, counts_pred, width, label='Predicted')
plt.xticks(x, class_names)
plt.ylabel('Number of images')
plt.title('True vs Predicted counts')
plt.legend()
plt.show()


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from torchvision.utils import make_grid


In [ ]:
# Map numeric labels to class names
class_names = ['non_autistic', 'autistic']

def imshow(inp, title=None):
    """Imshow for Tensor."""
    inp = inp.numpy().transpose((1, 2, 0))  # C x H x W -> H x W x C
    inp = inp * 0.5 + 0.5  # unnormalize from [-1,1] to [0,1]
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.axis('off')


In [ ]:
# Ensure model is in eval mode
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get one batch
dataiter = iter(test_loader)
images, labels = next(dataiter)
images, labels = images.to(device), labels.to(device)

# Get predictions
with torch.no_grad():
    outputs = model(images)
    _, preds = torch.max(outputs, 1)


In [ ]:
# Move images to CPU for plotting
images = images.cpu()
labels = labels.cpu()
preds = preds.cpu()

# Make a grid of images
img_grid = make_grid(images, nrow=4)
plt.figure(figsize=(12, 8))
imshow(img_grid, title=[f"T:{class_names[t]} | P:{class_names[p]}" for t,p in zip(labels, preds)])
plt.show()


In [ ]:
fig = plt.figure(figsize=(12, 8))
for i in range(min(16, len(images))):
    ax = fig.add_subplot(4, 4, i+1, xticks=[], yticks=[])
    img = images[i].numpy().transpose((1,2,0))
    img = img * 0.5 + 0.5  # unnormalize
    ax.imshow(img)
    ax.set_title(f"T:{class_names[labels[i]]}\nP:{class_names[preds[i]]}", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# Get the next batch from the test_loader
dataiter = iter(test_loader)
next(dataiter)  # Skip first batch (first 16)
images, labels = next(dataiter)  # Get the second batch (next 16)
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    outputs = model(images)
    _, preds = torch.max(outputs, 1)

# Move to CPU for plotting
images = images.cpu()
labels = labels.cpu()
preds = preds.cpu()


In [ ]:
fig = plt.figure(figsize=(12, 8))
for i in range(min(16, len(images))):
    ax = fig.add_subplot(4, 4, i+1, xticks=[], yticks=[])
    img = images[i].numpy().transpose((1,2,0))
    img = img * 0.5 + 0.5  # unnormalize
    ax.imshow(img)
    ax.set_title(f"T:{class_names[labels[i]]}\nP:{class_names[preds[i]]}", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
start_idx = 16
end_idx = 32
images, labels = zip(*[test_dataset[i] for i in range(start_idx, end_idx)])
images = torch.stack(images)
labels = torch.tensor(labels)

# Predict
images = images.to(device)
with torch.no_grad():
    outputs = model(images)
    _, preds = torch.max(outputs, 1)
preds = preds.cpu()


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


In [ ]:
# Map: 0 = non_autistic (normal), 1 = autistic (abnormal)
num_normal = all_preds.count(0)
num_abnormal = all_preds.count(1)

print(f"Normal  images predicted: {num_normal}")
print(f"Abnormal  images predicted: {num_abnormal}")
print(f"Total images: {len(all_preds)}")


In [ ]:
true_normal = all_labels.count(0)
true_abnormal = all_labels.count(1)

print(f"True normal images: {true_normal}")
print(f"True abnormal images: {true_abnormal}")


In [ ]:
import torch
import torch.nn.functional as F

def get_attention_rollout(model, x, discard_ratio=0.0):
    """
    Args:
        model: your ViTASD model (already loaded and eval mode)
        x: input image tensor (1, 3, 224, 224)
        discard_ratio: fraction of lowest attention values to discard
    Returns:
        rollout_map: 2D attention map (patch_size x patch_size)
    """
    attn_weights = []

    # Hook to capture attention from each block
    def hook_fn(module, input, output):
        # output[0] is the attention tensor
        attn_weights.append(output[0].detach())

    hooks = []
    for blk in model.backbone.blocks:
        h = blk.attn.attn_drop.register_forward_hook(hook_fn)
        hooks.append(h)

    _ = model(x)  # forward pass

    for h in hooks:
        h.remove()

    # Compute rollout
    rollout = torch.eye(attn_weights[0].size(-1)).to(x.device)
    for attn in attn_weights:
        attn_mean = attn.mean(dim=1)  # average over heads
        if discard_ratio > 0:
            flat = attn_mean.flatten(1)
            _, idx = flat.topk(int(flat.size(1)*(1-discard_ratio)), dim=-1)
            mask = torch.zeros_like(flat)
            mask.scatter_(1, idx, 1)
            attn_mean = (attn_mean * mask.view_as(attn_mean))
        rollout = rollout @ attn_mean  # multiply with previous layers

    # Skip CLS token
    mask = rollout[0, 1:]
    size = int(mask.size(0) ** 0.5)
    return mask.reshape(size, size)

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

def visualize_attention(image, attn_map):
    """
    Overlay attention map on image
    image: numpy array (H,W,3)
    attn_map: 2D tensor
    """
    attn_map = cv2.resize(attn_map.cpu().numpy(), (image.shape[1], image.shape[0]))
    heatmap = cv2.applyColorMap(np.uint8(255*attn_map), cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(image, 0.6, heatmap, 0.4, 0)
    plt.imshow(overlay)
    plt.axis('off')
    plt.show()

In [ ]:
train_dir = "/content/train"   # path to your training images folder
valid_dir = "/content/valid"   # path to your validation images folder
test_dir  = "/content/test"    # path to your test images folder

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms
from timm import create_model
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import cv2

# --- Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

image_size = 224
test_dir = '/content/drive/MyDrive/autism-image-dataset/test'
valid_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])
test_dataset = datasets.ImageFolder(test_dir, transform=valid_transform)
class_names = test_dataset.classes

# --- Model ---
class ViTASD(nn.Module):
    def __init__(self, num_classes=2):
        super(ViTASD, self).__init__()
        self.backbone = create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes)
    def forward(self, x):
        return self.backbone(x)

model = ViTASD().to(device)
model.load_state_dict(torch.load("/content/best_vit_autism_model.pth", map_location=device))
model.eval()

# --- Attention Rollout ---
def get_attention_rollout(model, img_tensor):
    model.eval()
    attentions = []

    def save_attention(module, input, output):
        attentions.append(output.detach())

    handles = [blk.attn.register_forward_hook(save_attention) for blk in model.backbone.blocks]

    with torch.no_grad():
        _ = model(img_tensor.unsqueeze(0).to(device))

    for h in handles:
        h.remove()

    att_mat = torch.stack(attentions).squeeze(1).mean(1)  # [layers, tokens, tokens]

    rollout = torch.eye(att_mat.size(-1)).to(att_mat.device)
    for a in att_mat:
        a = a + torch.eye(a.size(0)).to(a.device)
        a /= a.sum(dim=-1, keepdim=True)
        rollout = a @ rollout

    rollout = rollout[0, 1:]  # remove CLS token
    num_tokens = rollout.size(0)
    grid_size = int(np.ceil(np.sqrt(num_tokens)))
    pad = grid_size ** 2 - num_tokens
    rollout_padded = torch.cat([rollout, torch.zeros(pad, device=rollout.device)])
    attn_map = rollout_padded.reshape(grid_size, grid_size)
    return attn_map.cpu().numpy()

# --- Custom Blue→Green→Yellow→Red colormap overlay ---
def apply_custom_colormap(attn_map_norm):
    """
    Maps normalised [0,1] attention values to:
      0.00 – 0.25  →  Blue   (low attention)
      0.25 – 0.50  →  Green  (medium attention)
      0.50 – 0.75  →  Yellow/Orange (high attention)
      0.75 – 1.00  →  Red    (very high / critical)
    Returns an RGB image [H, W, 3] float32 in [0,1].
    """
    h, w = attn_map_norm.shape
    rgb = np.zeros((h, w, 3), dtype=np.float32)

    # --- Blue zone: 0.0 → 0.25 ---
    mask = attn_map_norm < 0.25
    t = attn_map_norm[mask] / 0.25          # 0→1 within zone
    rgb[mask, 0] = 0.0                       # R
    rgb[mask, 1] = t * 0.5                   # G  0→0.5
    rgb[mask, 2] = 1.0                       # B  full blue

    # --- Green zone: 0.25 → 0.50 ---
    mask = (attn_map_norm >= 0.25) & (attn_map_norm < 0.50)
    t = (attn_map_norm[mask] - 0.25) / 0.25  # 0→1 within zone
    rgb[mask, 0] = 0.0                        # R
    rgb[mask, 1] = 0.5 + t * 0.5             # G  0.5→1.0
    rgb[mask, 2] = 1.0 - t                    # B  1→0

    # --- Yellow/Orange zone: 0.50 → 0.75 ---
    mask = (attn_map_norm >= 0.50) & (attn_map_norm < 0.75)
    t = (attn_map_norm[mask] - 0.50) / 0.25  # 0→1 within zone
    rgb[mask, 0] = 0.5 + t * 0.5             # R  0.5→1.0
    rgb[mask, 1] = 1.0 - t * 0.5             # G  1.0→0.5
    rgb[mask, 2] = 0.0                        # B

    # --- Red zone: 0.75 → 1.00 ---
    mask = attn_map_norm >= 0.75
    t = (attn_map_norm[mask] - 0.75) / 0.25  # 0→1 within zone
    rgb[mask, 0] = 1.0                        # R  full red
    rgb[mask, 1] = 0.5 - t * 0.5             # G  0.5→0
    rgb[mask, 2] = 0.0                        # B

    return rgb

# --- Visualisation with labels, confidence, legend ---
def visualize_attention_rgb(img_tensor, attn_map, true_label, pred_label,
                            confidence, alpha=0.55):
    # Decode image
    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    # Resize & normalise attention map
    attn_resized = cv2.resize(attn_map, (img.shape[1], img.shape[0]),
                              interpolation=cv2.INTER_LINEAR)
    attn_norm = (attn_resized - attn_resized.min()) / \
                (attn_resized.max() - attn_resized.min() + 1e-8)

    # Build custom colormap heatmap
    heat_rgb = apply_custom_colormap(attn_norm)   # [H,W,3]

    # Blend: original image + heatmap
    overlay = (1 - alpha) * img + alpha * heat_rgb
    overlay = np.clip(overlay, 0, 1)

    # ── Plot ──────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    fig.patch.set_facecolor('#1a1a2e')

    for ax in axes:
        ax.set_facecolor('#1a1a2e')

    # Left: original image
    axes[0].imshow(img)
    axes[0].set_title("Original image", color='white', fontsize=12, pad=8)
    axes[0].axis('off')

    # Right: attention overlay
    axes[1].imshow(overlay)
    axes[1].set_title("Attention rollout (Blue→Green→Yellow→Red)",
                      color='white', fontsize=12, pad=8)
    axes[1].axis('off')

    # ── Colour legend ─────────────────────────────────────────────────────
    legend_patches = [
        mpatches.Patch(color=(0.0, 0.0, 1.0),   label='Blue  — low attention (0–25%)'),
        mpatches.Patch(color=(0.0, 1.0, 0.0),   label='Green — medium attention (25–50%)'),
        mpatches.Patch(color=(1.0, 0.75, 0.0),  label='Yellow/Orange — high attention (50–75%)'),
        mpatches.Patch(color=(1.0, 0.0, 0.0),   label='Red — critical attention (75–100%)'),
    ]
    axes[1].legend(
        handles=legend_patches,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.22),
        ncol=2,
        fontsize=9,
        framealpha=0.3,
        labelcolor='white',
        facecolor='#1a1a2e',
        edgecolor='gray'
    )

    # ── Header title ──────────────────────────────────────────────────────
    correct = true_label.lower() == pred_label.lower()
    status  = "CORRECT" if correct else "WRONG"
    title_color = '#00e676' if correct else '#ff5252'

    fig.suptitle(
        f"True: {true_label.upper()}   |   "
        f"Predicted: {pred_label.upper()}   |   "
        f"Confidence: {confidence:.1f}%   |   {status}",
        fontsize=13,
        fontweight='bold',
        color=title_color,
        y=1.01
    )

    plt.tight_layout()
    plt.show()
    print(f"  True: {true_label:20s} | Predicted: {pred_label:20s} | "
          f"Confidence: {confidence:.1f}% | {'✓ Correct' if correct else '✗ Wrong'}")


# --- Run on test set (predicted autistic only) ---
for idx, (img, true_idx) in enumerate(test_dataset):
    img_tensor = img.to(device)

    with torch.no_grad():
        output = model(img_tensor.unsqueeze(0))
        probs   = torch.softmax(output, dim=1)
        pred_idx   = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred_idx].item() * 100

    true_label = class_names[true_idx]
    pred_label = class_names[pred_idx]

    if pred_idx == 1:   # predicted autistic
        attn_map = get_attention_rollout(model, img_tensor)
        visualize_attention_rgb(img, attn_map, true_label, pred_label, confidence)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from timm import create_model
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, recall_score, precision_score,
                             roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 1. CONFIG
# ─────────────────────────────────────────────
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "/content/best_vit_autism_model.pth"
DATA_DIR   = "/content/drive/MyDrive/autism-image-dataset/test"
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_CLASSES = 2

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ─────────────────────────────────────────────
# 2. MODEL
# ─────────────────────────────────────────────
class ViTASD(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.backbone = create_model('vit_base_patch16_224',
                                     pretrained=False,
                                     num_classes=num_classes)
    def forward(self, x):
        return self.backbone(x)

def load_model():
    m = ViTASD().to(device)
    m.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    m.eval()
    return m

# ─────────────────────────────────────────────
# 3. EVALUATION ON ONE FOLD
# ─────────────────────────────────────────────
def evaluate_fold(model, loader):
    all_labels, all_probs, all_preds = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs   = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            preds   = torch.argmax(outputs, dim=1).cpu().numpy()
            all_labels.extend(labels.numpy())
            all_probs.extend(probs)
            all_preds.extend(preds)

    y_true = np.array(all_labels)
    y_prob = np.array(all_probs)
    y_pred = np.array(all_preds)

    return {
        'f1':        f1_score(y_true, y_pred,        zero_division=0),
        'recall':    recall_score(y_true, y_pred,    zero_division=0),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'auc_roc':   roc_auc_score(y_true, y_prob),
        'y_true':    y_true,
        'y_prob':    y_prob,
        'y_pred':    y_pred,
    }

# ─────────────────────────────────────────────
# 4. CROSS-VALIDATION RUNNER
# ─────────────────────────────────────────────
def run_cross_validation(k):
    print(f"\n{'='*60}")
    print(f"  {k}-FOLD CROSS VALIDATION")
    print(f"{'='*60}")

    dataset   = datasets.ImageFolder(DATA_DIR, transform=transform)
    labels    = [s[1] for s in dataset.samples]
    class_names = dataset.classes

    skf     = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    indices = np.arange(len(dataset))

    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(indices, labels), 1):
        print(f"\n  Fold {fold}/{k}  —  {len(test_idx)} test samples")

        test_subset = Subset(dataset, test_idx)
        test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE,
                                 shuffle=False, num_workers=2)

        model   = load_model()
        metrics = evaluate_fold(model, test_loader)
        fold_results.append(metrics)

        print(f"    F1        : {metrics['f1']:.4f}")
        print(f"    Recall    : {metrics['recall']:.4f}")
        print(f"    Precision : {metrics['precision']:.4f}")
        print(f"    AUC-ROC   : {metrics['auc_roc']:.4f}")

    return fold_results, class_names

# ─────────────────────────────────────────────
# 5. SUMMARY TABLE
# ─────────────────────────────────────────────
def print_summary(fold_results, k):
    metrics_keys = ['f1', 'recall', 'precision', 'auc_roc']
    labels_map   = {'f1': 'F1 Score', 'recall': 'Recall',
                    'precision': 'Precision', 'auc_roc': 'AUC-ROC'}

    print(f"\n{'─'*60}")
    print(f"  {k}-Fold Summary")
    print(f"{'─'*60}")
    print(f"  {'Metric':<15} {'Mean':>8}  {'Std':>8}  {'Min':>8}  {'Max':>8}")
    print(f"  {'─'*55}")

    summary = {}
    for key in metrics_keys:
        vals = [r[key] for r in fold_results]
        summary[key] = {'mean': np.mean(vals), 'std': np.std(vals),
                        'min': np.min(vals),   'max': np.max(vals),
                        'vals': vals}
        print(f"  {labels_map[key]:<15} "
              f"{np.mean(vals):>8.4f}  "
              f"{np.std(vals):>8.4f}  "
              f"{np.min(vals):>8.4f}  "
              f"{np.max(vals):>8.4f}")

    return summary

# ─────────────────────────────────────────────
# 6. PLOTS
# ─────────────────────────────────────────────
def plot_metrics_bar(summary3, summary5):
    metrics    = ['f1', 'recall', 'precision', 'auc_roc']
    labels_map = {'f1': 'F1 Score', 'recall': 'Recall',
                  'precision': 'Precision', 'auc_roc': 'AUC-ROC'}

    means3 = [summary3[m]['mean'] for m in metrics]
    stds3  = [summary3[m]['std']  for m in metrics]
    means5 = [summary5[m]['mean'] for m in metrics]
    stds5  = [summary5[m]['std']  for m in metrics]

    x     = np.arange(len(metrics))
    width = 0.35

    fig, ax = plt.subplots(figsize=(11, 6))
    bars3 = ax.bar(x - width/2, means3, width, yerr=stds3,
                   label='3-Fold CV', color='#4C72B0', capsize=5,
                   error_kw={'linewidth': 1.5})
    bars5 = ax.bar(x + width/2, means5, width, yerr=stds5,
                   label='5-Fold CV', color='#DD8452', capsize=5,
                   error_kw={'linewidth': 1.5})

    # Value labels on bars
    for bar in bars3:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
    for bar in bars5:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

    ax.set_xlabel('Metric', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('3-Fold vs 5-Fold Cross Validation — Metric Comparison', fontsize=13)
    ax.set_xticks(x)
    ax.set_xticklabels([labels_map[m] for m in metrics], fontsize=11)
    ax.set_ylim(0, 1.1)
    ax.legend(fontsize=11)
    ax.yaxis.grid(True, alpha=0.4)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.savefig('cv_metric_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_roc_curves(results3, results5):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for ax, results, k, color in zip(
            axes, [results3, results5], [3, 5], ['#4C72B0', '#DD8452']):

        all_auc = []
        for i, r in enumerate(results, 1):
            fpr, tpr, _ = roc_curve(r['y_true'], r['y_prob'])
            auc_val = r['auc_roc']
            all_auc.append(auc_val)
            ax.plot(fpr, tpr, alpha=0.55, lw=1.5,
                    label=f'Fold {i} (AUC={auc_val:.3f})')

        ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1.02])
        ax.set_xlabel('False Positive Rate', fontsize=11)
        ax.set_ylabel('True Positive Rate', fontsize=11)
        ax.set_title(f'{k}-Fold ROC Curves\n'
                     f'Mean AUC = {np.mean(all_auc):.4f} ± {np.std(all_auc):.4f}',
                     fontsize=12)
        ax.legend(fontsize=8, loc='lower right')
        ax.grid(alpha=0.3)

    plt.suptitle('ROC Curves — 3-Fold vs 5-Fold Cross Validation',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('cv_roc_curves.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_fold_scores(summary3, summary5):
    metrics    = ['f1', 'recall', 'precision', 'auc_roc']
    labels_map = {'f1': 'F1 Score', 'recall': 'Recall',
                  'precision': 'Precision', 'auc_roc': 'AUC-ROC'}

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    axes = axes.flatten()

    for i, metric in enumerate(metrics):
        ax = axes[i]
        vals3 = summary3[metric]['vals']
        vals5 = summary5[metric]['vals']

        ax.plot(range(1, 4), vals3, 'o-', color='#4C72B0',
                label='3-Fold', lw=2, markersize=7)
        ax.plot(range(1, 6), vals5, 's-', color='#DD8452',
                label='5-Fold', lw=2, markersize=7)

        ax.axhline(np.mean(vals3), color='#4C72B0',
                   linestyle='--', alpha=0.5, lw=1)
        ax.axhline(np.mean(vals5), color='#DD8452',
                   linestyle='--', alpha=0.5, lw=1)

        ax.set_title(labels_map[metric], fontsize=12)
        ax.set_xlabel('Fold', fontsize=10)
        ax.set_ylabel('Score', fontsize=10)
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

    plt.suptitle('Per-Fold Scores — 3-Fold vs 5-Fold',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('cv_per_fold_scores.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_confusion_matrices(results3, results5, class_names):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ax, results, k in zip(axes, [results3, results5], [3, 5]):
        # Aggregate all folds
        y_true_all = np.concatenate([r['y_true'] for r in results])
        y_pred_all = np.concatenate([r['y_pred'] for r in results])
        cm = confusion_matrix(y_true_all, y_pred_all)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                      display_labels=class_names)
        disp.plot(ax=ax, colorbar=False, cmap='Blues')
        ax.set_title(f'{k}-Fold Aggregated Confusion Matrix', fontsize=12)

    plt.suptitle('Confusion Matrices — All Folds Combined',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('cv_confusion_matrices.png', dpi=150, bbox_inches='tight')
    plt.show()

# ─────────────────────────────────────────────
# 7. MAIN — RUN EVERYTHING
# ─────────────────────────────────────────────
results3, class_names = run_cross_validation(k=3)
summary3 = print_summary(results3, k=3)

results5, _           = run_cross_validation(k=5)
summary5 = print_summary(results5, k=5)

# ── Plots ────────────────────────────────────
plot_metrics_bar(summary3, summary5)
plot_roc_curves(results3, results5)
plot_fold_scores(summary3, summary5)
plot_confusion_matrices(results3, results5, class_names)

print("\nAll plots saved to current directory.")

In [ ]:
import os
os.chdir('/content/my_app')

In [ ]:
import os
for f in os.listdir():
    print(f)

In [ ]:
import os
print(os.listdir('/content'))

In [ ]:
import os
for f in os.listdir():
    print(f)

In [ ]:
!git add .